**DATA PREPARATION AND EXPLORATORY ANALYSIS**

Initial Inspection

In [1]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import pandas as pd

df = pd.read_csv(
    "/content/drive/MyDrive/household_power_consumption.txt",
    sep=";",
    na_values=["?"],
    low_memory=False
)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/household_power_consumption.txt'

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print(df.isnull().sum())


In [ ]:
print(df.isnull().mean() * 100)

In [ ]:
df.duplicated().sum()

**Commentary:**
 The dataset contains 913,884 observations across 9 columns. Seven of them are numerical measurements stored as float64, while Date and Time are stored as object types, not datetime. They will need to be parsed into a proper datetime format before any time-based analysis can be done reliably.

Each of the seven measurement columns has 3,948 missing values (0.432% of records), the Time column has one missing value, and the Date column is complete. No duplicate records were found. Even though the share of missing data is small, it must be addressed before modelling: most algorithms cannot handle null values and will either fail or silently drop the affected rows, which can break the continuity of a time series and bias the results. Because all seven columns have the same number of gaps, they likely occur in the same rows, which points to whole-record outages in data collection rather than isolated sensor faults. This should be checked before choosing between removing the rows and imputing them.

Global Active Power averages approximately 1.091 kW and Voltage approximately 239.84 V. The sub-metering columns have low averages compared with their maximum values, which suggests intermittent consumption: long stretches of low use interrupted by occasional high-demand events. Dispersion varies across the measurements, so the variables sit on different scales and may need scaling or normalisation for some models. Some maximum values also suggest potential outliers. These have not been removed or treated, and they need further investigation to determine whether they reflect genuine peak usage or measurement errors.

Exploratory Data Analysis

In [ ]:
df[df["Time"].isnull()]

In [ ]:
df = df.dropna(subset=["Time"])

In [ ]:
df.shape

In [ ]:
df["DateTime"] = pd.to_datetime(
    df["Date"] + " " + df["Time"],
    format="%d/%m/%Y %H:%M:%S"
)

In [ ]:
df = df.dropna(subset=["Time"]).copy()

In [ ]:
df["DateTime"] = pd.to_datetime(
    df["Date"] + " " + df["Time"],
    format="%d/%m/%Y %H:%M:%S"
)

In [ ]:
df[["Date", "Time", "DateTime"]].head()

In [ ]:
df = df.sort_values("DateTime")

In [ ]:
df[["DateTime", "Global_active_power"]].head()

In [ ]:
df[["DateTime", "Global_active_power"]].tail()

In [ ]:
df = df.set_index("DateTime")

In [ ]:
df.head()

In [ ]:
hourly_df = df.select_dtypes(include="number").resample("h").mean()

In [ ]:
hourly_df.head()

In [ ]:
hourly_df.shape

In [ ]:
hourly_df.isnull().sum()

Histogram

In [ ]:
import matplotlib.pyplot as plt

hourly_df.hist(figsize=(15, 12), bins=30)

plt.tight_layout()
plt.show()

In [ ]:
hourly_df.skew()

In [ ]:
hourly_df.describe().loc[["mean", "50%"]]

**Interpretation:**

Boxplot

In [ ]:
hourly_df.plot(
    kind="box",
    subplots=True,
    layout=(4, 2),
    figsize=(15, 14),
    sharex=False,
    sharey=False
)

plt.tight_layout()
plt.show()

In [ ]:
Q1 = hourly_df.quantile(0.25)
Q3 = hourly_df.quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = (
    (hourly_df < lower_bound) |
    (hourly_df > upper_bound)
).sum()

outliers

In [ ]:
(outliers / len(hourly_df) * 100).round(2)

**Interpretation:**

Correlation heatmap

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

correlation_matrix = hourly_df.corr()

plt.figure(figsize=(10, 8))

sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Heatmap of Household Power Consumption")
plt.tight_layout()
plt.show()

In [ ]:
correlation_matrix.loc[
    "Global_active_power",
    "Global_intensity"
]

**Interpretation:**

Time series Plot

In [ ]:
plt.figure(figsize=(15, 5))

plt.plot(
    hourly_df.index,
    hourly_df["Global_active_power"]
)

plt.title("Hourly Household Electricity Consumption Over Time")
plt.xlabel("Date")
plt.ylabel("Global Active Power (kW)")
plt.tight_layout()
plt.show()

In [ ]:
monthly_consumption = hourly_df["Global_active_power"].resample("ME").mean()

monthly_consumption.head(10)

In [ ]:
monthly_consumption.describe()

In [ ]:
monthly_consumption.tail(12)

In [ ]:
hourly_df["Global_active_power"].resample("ME").count().tail(3)

**Interpretation:**

**Data Cleaning**

In [ ]:
# Check the number of missing values in each column
hourly_df.isnull().sum()

In [ ]:
# Display the timestamps where Global_active_power is missing
hourly_df[hourly_df["Global_active_power"].isnull()]

In [ ]:
# Identify consecutive periods with missing observations
missing_times = hourly_df.index[
    hourly_df["Global_active_power"].isnull()
]

missing_gaps = missing_times.to_series().diff()

print("Missing timestamps:", len(missing_times))
print("Time differences between missing timestamps:")
print(missing_gaps.value_counts())

In [ ]:
# Remove hourly observations with missing values
hourly_df = hourly_df.dropna().copy()

# Check the remaining missing values
hourly_df.isnull().sum()

In [ ]:
hourly_df.shape

In [ ]:
print("Remaining missing values:")
print(hourly_df.isnull().sum().sum())

print("Remaining duplicate rows:")
print(hourly_df.duplicated().sum())

In [ ]:
# Examine the minimum and maximum values
hourly_df.agg(["min", "max"]).T

In [ ]:
# Count negative values in each numerical variable
(hourly_df < 0).sum()

**Commentary:**

**Feature Engineering and Preprocessing**

In [ ]:
# Check the time differences between consecutive observations
hourly_df.index.to_series().diff().value_counts().head(10)

In [ ]:
# Restore the complete hourly time index
full_hourly_index = pd.date_range(
    start=hourly_df.index.min(),
    end=hourly_df.index.max(),
    freq="h"
)

hourly_df = hourly_df.reindex(full_hourly_index)

hourly_df.index.name = "DateTime"

# Check missing values after restoring the timeline
hourly_df.isnull().sum()

In [ ]:
hourly_df["Hour"] = hourly_df.index.hour

hourly_df[["Hour"]].head()

In [ ]:
hourly_df["DayOfWeek"] = hourly_df.index.dayofweek

hourly_df[["DayOfWeek"]].head()

In [ ]:
hourly_df["Month"] = hourly_df.index.month

hourly_df[["Month"]].head()

In [ ]:
hourly_df["IsWeekend"] = (
    hourly_df.index.dayofweek >= 5
).astype(int)

hourly_df[["IsWeekend"]].head()

In [ ]:
hourly_df[
    ["Hour", "DayOfWeek", "Month", "IsWeekend"]
].head(10)

In [ ]:
for lag in range(1, 25):
    hourly_df[f"Lag_{lag}"] = (
        hourly_df["Global_active_power"].shift(lag)
    )

lag_columns = [f"Lag_{i}" for i in range(1, 25)]

display(
    hourly_df[
        ["Global_active_power"] + lag_columns
    ].head(30)
)

In [ ]:
print("Number of lag features:",
      len(lag_columns))

print("\nMissing values in each lag:")
print(hourly_df[lag_columns].isnull().sum())

Interpretation:

**Feature Engineering and Preprocessing**

In [ ]:
for window in [3, 6, 12, 24]:
    hourly_df[f"Rolling_Mean_{window}"] = (
        hourly_df["Global_active_power"]
        .shift(1)
        .rolling(window=window)
        .mean()
    )

In [ ]:
rolling_columns = [
    "Rolling_Mean_3",
    "Rolling_Mean_6",
    "Rolling_Mean_12",
    "Rolling_Mean_24"
]

display(
    hourly_df[
        ["Global_active_power"] + rolling_columns
    ].head(30)
)

In [ ]:
for window in [3, 6, 12, 24]:
    hourly_df[f"Rolling_Std_{window}"] = (
        hourly_df["Global_active_power"]
        .shift(1)
        .rolling(window=window)
        .std()
    )

In [ ]:
rolling_std_columns = [
    "Rolling_Std_3",
    "Rolling_Std_6",
    "Rolling_Std_12",
    "Rolling_Std_24"
]

display(
    hourly_df[
        ["Global_active_power"] + rolling_std_columns
    ].head(30)
)

In [ ]:
print("Dataset shape:", hourly_df.shape)

print("\nMissing values in each column:")
print(hourly_df.isnull().sum())

In [ ]:
missing_times = hourly_df.index[
    hourly_df["Global_active_power"].isna()
]

missing_series = pd.Series(missing_times, index=missing_times)

groups = missing_series.diff().ne(
    pd.Timedelta(hours=1)
).cumsum()

print("Missing hourly records:", len(missing_times))
print("\nMissing time intervals:")

for _, group in missing_series.groupby(groups):
    print(
        group.iloc[0],
        "to",
        group.iloc[-1],
        "| Hours:",
        len(group)
    )

In [ ]:
# Interpolate the isolated missing hour
hourly_df = hourly_df.interpolate(
    method="time",
    limit=1,
    limit_area="inside"
)

# Remove rows that still contain missing original measurements
measurement_columns = [
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3"
]

hourly_df = hourly_df.dropna(
    subset=measurement_columns
).copy()

print("Dataset shape:", hourly_df.shape)

print("\nMissing values in original measurements:")
print(hourly_df[measurement_columns].isnull().sum())

In [ ]:
# Recreate the complete hourly timeline
full_index = pd.date_range(
    start=hourly_df.index.min(),
    end=hourly_df.index.max(),
    freq="h"
)

# Restore missing hours in the timeline
target_full = hourly_df["Global_active_power"].reindex(full_index)

# Recalculate lag features
for lag in range(1, 25):
    hourly_df[f"Lag_{lag}"] = (
        target_full.shift(lag).reindex(hourly_df.index)
    )

# Recalculate rolling means and standard deviations
for window in [3, 6, 12, 24]:

    hourly_df[f"Rolling_Mean_{window}"] = (
        target_full.shift(1)
        .rolling(window=window)
        .mean()
        .reindex(hourly_df.index)
    )

    hourly_df[f"Rolling_Std_{window}"] = (
        target_full.shift(1)
        .rolling(window=window)
        .std()
        .reindex(hourly_df.index)
    )

print("Updated dataset shape:", hourly_df.shape)

print("\nMissing values in engineered features:")
engineered_columns = (
    [f"Lag_{i}" for i in range(1, 25)]
    + [f"Rolling_Mean_{i}" for i in [3, 6, 12, 24]]
    + [f"Rolling_Std_{i}" for i in [3, 6, 12, 24]]
)

print(hourly_df[engineered_columns].isnull().sum())

In [ ]:
# Remove rows with incomplete lag and rolling features
hourly_df = hourly_df.dropna(
    subset=engineered_columns
).copy()

print("Final dataset shape:", hourly_df.shape)

print("\nRemaining missing values:")
print(hourly_df.isnull().sum())

print("\nDuplicate timestamps:", hourly_df.index.duplicated().sum())

print("\nChronological order:", hourly_df.index.is_monotonic_increasing)

In [ ]:
# Define the target variable
y = hourly_df["Global_active_power"]

# Define the predictor variables
feature_columns = (
    ["Hour", "DayOfWeek", "Month", "IsWeekend"]
    + [f"Lag_{i}" for i in range(1, 25)]
    + [f"Rolling_Mean_{i}" for i in [3, 6, 12, 24]]
    + [f"Rolling_Std_{i}" for i in [3, 6, 12, 24]]
)

X = hourly_df[feature_columns]

print("Feature dataset shape:", X.shape)
print("Target dataset shape:", y.shape)

print("\nTarget variable:", y.name)
print("\nNumber of predictor features:", len(feature_columns))

Interpretation:

**Train/Test Split**

In [ ]:
# Determine the 80% training split point
split_index = int(len(X) * 0.80)

# Split the features
X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

# Split the target
y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining period:")
print(X_train.index.min(), "to", X_train.index.max())

print("\nTesting period:")
print(X_test.index.min(), "to", X_test.index.max())

Interpretation:

**Pipeline Construction:**

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Identify numerical features
numeric_features = feature_columns

# Create the column transformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features)
    ]
)

# Create the preprocessing pipeline
preprocessing_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor)
    ]
)

print("Preprocessing pipeline created successfully.")

In [ ]:
# Fit on training data and transform it
X_train_processed = preprocessing_pipeline.fit_transform(X_train)

# Transform testing data without fitting again
X_test_processed = preprocessing_pipeline.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

In [ ]:
import pandas as pd

# Convert processed training data back to a DataFrame
X_train_scaled = pd.DataFrame(
    X_train_processed,
    columns=feature_columns,
    index=X_train.index
)

# Display the mean and standard deviation
print("Feature means after scaling:")
print(X_train_scaled.mean().round(3))

print("\nFeature standard deviations after scaling:")
print(X_train_scaled.std().round(3))

In [ ]:
# Save the final prepared dataset
hourly_df.to_csv(
    "final_household_power_dataset.csv",
    index=True
)

print("Dataset saved successfully!")